# Colab-start: repo ophalen, packages installeren en features inlezen

**Datalab 3 · werkt in Google Colab én lokaal**

Dit notebook is het startpunt voor werk in Colab, zoals de grid search van XGBoost. Het doet drie dingen:

1. **Omgeving bepalen:** draait het notebook in Colab of lokaal?
2. **Repo en packages:** in Colab wordt de repo gecloned (of bijgewerkt) en worden de packages uit `pyproject.toml` geïnstalleerd. Lokaal regelt `uv sync` dit al.
3. **Features inlezen:** `bee_features_per_second.csv` laden, opschonen en ontdubbelen, met dezelfde stappen als de EDA.

**Openen in Colab:** `https://colab.research.google.com/github/Khaleellll/Datalab-lll/blob/main/notebooks/colab_start.ipynb`
(zolang dit notebook nog niet op `main` staat: vervang `main` door de naam van de branch).

> **AVG:** de WAV-opnames bevatten stemmen. Zet ze **niet** in Google Drive voordat Sam akkoord heeft gegeven (Malak vraagt dit na).
> Het featurebestand bevat alleen getallen per seconde en geen audio. Voor modellen op de huidige features is alleen dat bestand nodig.

## 1. Omgeving bepalen

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print("Omgeving:", "Google Colab" if IN_COLAB else "lokaal")

## 2. Repo ophalen en packages installeren (alleen Colab)

- De repo wordt gecloned naar `/content/Datalab-lll`. Bestaat die map al (tweede keer draaien in dezelfde sessie), dan wordt hij bijgewerkt met `git pull`.
- Wil je op een andere branch werken, vul dan `BRANCH` in.
- Packages installeren we met uv uit `pyproject.toml`, net als lokaal. Colab heeft al veel packages; uv voegt alleen toe wat ontbreekt of te oud is, zoals `xgboost` en `librosa`.

Lokaal sla je deze cel over: daar heb je `uv sync` al gedraaid.

In [ ]:
REPO_URL = "https://github.com/Khaleellll/Datalab-lll.git"
BRANCH = "main"  # bijvoorbeeld "taak/xgboost-colab"

if IN_COLAB:
    import os
    import subprocess

    REPO_DIR = Path("/content/Datalab-lll")
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "switch", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", str(REPO_DIR / "pyproject.toml")], check=True)
    os.chdir(REPO_DIR / "notebooks")  # zelfde werkmap als lokaal: paden beginnen met ../data/
    print("Repo klaar in", REPO_DIR, "| branch:", BRANCH)
else:
    print("Lokaal: niets te doen. Werkmap:", Path.cwd())

Controle: zijn de belangrijkste packages er, en welke versies?

In [ ]:
import librosa
import numpy as np
import pandas as pd
import sklearn
import xgboost

for naam, module in [("numpy", np), ("pandas", pd), ("scikit-learn", sklearn), ("xgboost", xgboost), ("librosa", librosa)]:
    print(f"{naam:<13} {module.__version__}")

## 3. Data koppelen

De mappen heten hetzelfde als lokaal (zie `data/README.md`):

```
data/
├── raw_data/          # WAV-opnames (alleen lokaal, zie AVG-opmerking)
├── raw_labels/        # buzzdetection_labels.csv
└── processed_data/    # bee_features_per_second.csv
```

- **Lokaal:** de data staat in `../data/`. Je hoeft niets te doen.
- **Colab:** zet in je eigen Google Drive een map `Datalab-lll-data` met dezelfde submappen (in elk geval `processed_data/` en `raw_labels/`). De cel hieronder koppelt Drive en legt een snelkoppeling van `data/` naar die map. Zo werken alle paden met `../data/` in beide omgevingen.

Staat jouw map ergens anders in Drive? Pas dan `DRIVE_DATA` aan.

In [ ]:
DRIVE_DATA = "/content/drive/MyDrive/Datalab-lll-data"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    bron = Path(DRIVE_DATA)
    if not bron.exists():
        raise FileNotFoundError(f"Map {bron} niet gevonden in Google Drive. Maak hem aan of pas DRIVE_DATA aan.")
    doel = Path("../data")
    for submap in ["raw_data", "raw_labels", "processed_data"]:
        if (bron / submap).exists() and not (doel / submap).exists():
            (doel / submap).symlink_to(bron / submap)
    print("Gekoppeld:", sorted(p.name for p in doel.iterdir()))

## 4. Features inlezen

`laad_features()` doet dezelfde opschoning als de EDA:

- spaties rond kolomnamen en tekstwaarden weghalen;
- **ontdubbelen:** `preprocessdata.py` leest op Windows en macOS elk WAV-bestand twee keer in, waardoor elke rij dubbel staat (bekend probleem, staat op de planning van Khaleel). Als het script gerepareerd is, verandert hier niets;
- `recorder` (16 kHz of 384 kHz) toevoegen, als de WAV-bestanden er zijn. Zonder WAV's blijft die kolom leeg.

In [ ]:
DATA_DIR = Path("../data")
FEATURES_CSV = DATA_DIR / "processed_data" / "bee_features_per_second.csv"
RAW_DATA_DIR = DATA_DIR / "raw_data"
TEKST_KOLOMMEN = ["start", "end", "label", "source_file"]


def laad_features(pad=FEATURES_CSV, raw_dir=RAW_DATA_DIR):
    # Featurebestand inlezen, opschonen en ontdubbelen. Geeft (df, lijst met featurekolommen).
    df = pd.read_csv(pad, skipinitialspace=True)
    df.columns = df.columns.str.strip()
    for kolom in TEKST_KOLOMMEN:
        df[kolom] = df[kolom].str.strip()

    voor = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Ingelezen: {voor} rijen, na ontdubbelen: {len(df)}")

    wavs = list(Path(raw_dir).glob("*.WAV")) + list(Path(raw_dir).glob("*.wav")) if Path(raw_dir).exists() else []
    if wavs:
        import soundfile as sf

        samplerate = {p.name: sf.info(p).samplerate for p in wavs}
        df["recorder"] = df["source_file"].map(samplerate).map({16000: "16 kHz", 384000: "384 kHz"})
    else:
        df["recorder"] = pd.NA
        print("Geen WAV-bestanden gevonden: kolom 'recorder' blijft leeg.")

    features = [k for k in df.columns if k not in TEKST_KOLOMMEN + ["recorder"]]
    return df, features


df, FEATURES = laad_features()
print("Aantal features:", len(FEATURES))

## 5. Controle

Klopt de data met wat we uit de EDA weten? Verwacht: 9.198 unieke seconden, 64 features, 3.716 gelabelde seconden, waarvan 139 `buzz` en 111 `flight`.

In [ ]:
print(f"Rijen: {len(df)} | bestanden: {df['source_file'].nunique()} | features: {len(FEATURES)}")
print(f"Gelabeld: {df['label'].notna().sum()} | zonder label: {df['label'].isna().sum()} ({df['label'].isna().mean():.1%})")
print()
print(df["label"].value_counts().rename("seconden"))
if df["recorder"].notna().any():
    print()
    print(pd.crosstab(df["label"].fillna("(geen label)"), df["recorder"]))

## 6. Klaar voor modelleren

Gelabelde data, klassen samengevoegd tot `buzz`, `flight` en `ruis` (zoals in `hommel_classificatie.ipynb`), en de groepen voor de split per opname (`StratifiedGroupKFold` op `source_file`, zie `docs/planning.md`).

In [ ]:
lab = df[df["label"].notna()].copy()
lab["klasse"] = lab["label"].where(lab["label"].isin(["buzz", "flight"]), "ruis")

X = lab[FEATURES]
y = lab["klasse"]
groepen = lab["source_file"]

print("X:", X.shape)
print(y.value_counts().rename("seconden"))
print("Bestanden met hommelgeluid:", lab.loc[y != "ruis", "source_file"].nunique())